# SPR-PCF Studio: Interactive Research Tutorial

Welcome to **SPR-PCF Studio**, a universal computational framework for designing, simulating, and analyzing surface plasmon resonance (SPR) photonic crystal fiber biosensors.

This tutorial demonstrates:
1. Creating parametric PCF geometries
2. Inspecting optical dispersion models (Sellmeier & Drude-Lorentz)
3. Generating and refining finite element meshes
4. Full-vector frequency-domain Maxwell mode solving
5. Extracting modal confinement loss and SPR resonance
6. Biosensing performance evaluation: Wavelength Sensitivity, FOM, and Resolution

In [ ]:
import sprpcf as spr
print(f"Loaded SPR-PCF Studio version: {spr.__version__}")

## 1. Design D-Shaped SPR-PCF Sensor Geometry

In [ ]:
project = spr.Project("Interactive_SPR_Sensor")

geometry = spr.DShapedPCF(
    pitch=2.0e-6,
    rings=2,
    hole_radius=0.5e-6,
    polish_depth=3.2e-6,
    metal_thickness=40e-9,
    cladding_radius=6.5e-6
)
geometry.configure_pml(thickness=1.2e-6)
project.set_geometry(geometry)

project.assign_material("cladding", "silica")
project.assign_material("fiber_core", "silica")
project.assign_material("plasmonic_metal_film", "gold")
project.assign_material("sensing_analyte_channel", "analyte")

fig = spr.plot_geometry(project)
fig.show()

## 2. Mesh Generation

In [ ]:
mesh = project.generate_mesh(backend="native", global_size=0.5e-6)
report = spr.validate_mesh(mesh)
print(f"Elements: {mesh.num_elements}, Nodes: {mesh.num_nodes}, Mean Quality: {report.mean_quality:.3f}")

fig_mesh = spr.plot_mesh(mesh, quality_map=True)
fig_mesh.show()

## 3. Full-Vector Wavelength Sweep Simulation

In [ ]:
study = spr.WavelengthSweepStudy(
    start_wavelength_m=850e-9,
    stop_wavelength_m=1350e-9,
    num_points=11,
    modes=2
)
study.set_solver(backend="native", grid_points=50, tolerance=1e-6)
project.add_study(study)

results = project.run()
print(f"Completed {len(results)} wavelength steps.")

## 4. SPR Resonance & Confinement Loss Analysis

In [ ]:
analyzer = spr.SPRAnalyzer(results.results)
loss_spec = analyzer.extract_loss_spectrum(mode_idx=0)
peaks = spr.extract_resonance_peaks(loss_spec.wavelengths_m, loss_spec.loss_db_per_cm)

if peaks:
    peak = peaks[0]
    print(f"Resonance Wavelength: {peak.wavelength_nm:.2f} nm")
    print(f"Peak Confinement Loss: {peak.peak_loss_db_cm:.2f} dB/cm")
    print(f"FWHM Linewidth:        {peak.fwhm_nm:.2f} nm")

fig_loss = spr.plot_loss_spectrum(analyzer)
fig_loss.show()

## 5. Mode Field Visualization

In [ ]:
mid_res = results.results[len(results) // 2]
mode = mid_res.modes[0]

fig_field = spr.plot_mode_field(mode, component="E_mag")
fig_field.show()